# Flash Drought Detection

In this notebook, we will apply the technique we learned in the previous part to a new case study in South Africa in 2024 and determine its predictive skill as an early warning of drought. In particular, we will apply a "detection rule" to thje time series of SIF-RCI and SWDI to see when our data would indicate a developing flash drought. The "detection rule" is based on two primary values that are chosen subjectively:

* The **drought severity**: a threshold value below which we declare a drought
* The **speed of drought development**: the length of time that an indicator changes from normal/mild to severe anomaly is a distinguishing factor for flash drought

In [ ]:
"""Inputs
In this case study, we will use a portion of Free State in South Africa as a region of interest, but this notebook
is set up to make it easier for you to input your own region of interest and date range. Along with modifying the input values
in this cell, you will want to run the Appendix notebook with updated values.

Directions for using your own region:
1. Using GIS software or Google Maps, determine the coordinates of a region of interest. Currently, only rectangular regions are supported.
2. Go to the Appendix notebook (3_appendix.ipynb), run the first code cell, then follow the instructions in the second code cell.
3. Find the crop calendar for the primary crop(s) in your region of interest. This will require your own research or prior knowledge, but it is
  very important to determine if a flash drought signal is agriculturally impactful.
4. Set the following variables in this cell:
  a. region_name, plot_region_name (str): Any name you want. region_name will be used in the filenames, plot_region_name will be used in the plot title.
  b. west, south, east, north (four floats): The coordinates of the region of interest.
  c. date_range (list[datetime]): The start and end dates of the time period you want to analyze.
  d. planting_start, planting_end, harvest_start, harvest_end (four datetimes): The crop calendar dates for the region.
  e. plot_title (str): Title to be used in the plot in the fourth cell.
"""
from datetime import datetime, timedelta

# Free State South Africa region of interest
#region_name = "south_africa"
#plot_region_name = "Free State South Africa"
# 28.00°–29.00°S, 27.00°–28.50°E
#west, south, east, north = 27.0, -29.0, 28.5, -28.0
# Northern Great Plains region of interest
region_name = "ngreat_plains"
plot_region_name = "Northern Great Plains"
# 45.00°–50.00°N, 106.00°–111.00°W
west, south, east, north = -111.0, 45.0, -106.0, 50.0
# For the South Africa case study, we will download data from September 2023 (just before the spring planting) through to
# August 2024 (after the harvest)
#date_range = [datetime(2023, 9, 1), datetime(2024, 8, 31)] # noqa: DTZ001
date_range = [datetime(2024, 3, 1), datetime(2024, 10, 31)] # noqa: DTZ001
start, end = date_range

# Typical crop calendar for Maize (Corn) in Free State South Africa:
#   Planting: October 15 – November 15
#   Harvest:  April 1 – May 15
#planting_start, planting_end = datetime(start.year, 10, 15), datetime(start.year, 11, 15) # noqa: DTZ001
#harvest_start, harvest_end = datetime(end.year, 4, 1), datetime(end.year, 5, 15) # noqa: DTZ001
planting_start, planting_end = datetime(start.year, 4, 13), datetime(start.year, 5, 13) # noqa: DTZ001
harvest_start, harvest_end = datetime(end.year, 8, 1), datetime(end.year, 9, 1) # noqa: DTZ001

years = sorted({str(dt.year) for dt in date_range})
year_str = '_'.join(years)
plot_year_str = '-'.join(years)

def format_coords(coord_left: float, coord_right: float, is_lat: bool = True) -> str:
    if is_lat:
        hem_left  = "N" if coord_left > 0.0 else "S"
        hem_right = "N" if coord_right > 0.0 else "S"
    else:
        hem_left  = "E" if coord_left > 0.0 else "W"
        hem_right = "E" if coord_right > 0.0 else "W"
    hem_left = "" if hem_right == hem_left else hem_left
    return f"{abs(coord_left)}°{hem_left}-{abs(coord_right)}°{hem_right}"

plot_title = f"SIF-RCI over {plot_region_name} ({plot_year_str})\n{format_coords(north, south)}, {format_coords(west, east, is_lat=False)}"

def doy_to_date(year, doy):
    return datetime(year, 1, 1) + timedelta(days=doy - 1) # noqa: DTZ001

doy_range: list[tuple[int, int]] = []
for year in range(start.year, end.year + 1):
    for doy in range(1, 365, 8):
        d = doy_to_date(year, doy)
        if start <= d <= end:
            doy_range.append((year, doy))
print(doy_range)

# Set the filename to use for the time series
sif_time_series_fname = f"{region_name}_{year_str}_sif.csv"
swdi_time_series_fname = f"{region_name}_{year_str}_swdi.csv"

In [ ]:
import os

from download import download_unpack_gosif
from tqdm.notebook import tqdm

output_dir = "data/gosif"
os.makedirs(output_dir, exist_ok=True)

gosif_geotiffs: list[str] = []
for year, doy in tqdm(doy_range, desc="Downloading granules"):
    fname = download_unpack_gosif(
        year,
        day=doy,
        output_dir=output_dir,
        verbose=False
    )
    if fname:
        gosif_geotiffs.append(fname)

In [ ]:
from analysis import compute_sif_time_series, compute_swdi_timeseries

# Get the climatology inputs produced by the appendix notebook
#clim_dir = f"inputs/sif_increments_{region_name}/"
clim_dir = "inputs/sif_increments/"

if not os.path.exists(f"data/{sif_time_series_fname}"):
    sif_csv_path, n_dates = compute_sif_time_series(
        gosif_geotiffs,
        clim_dir,
        sif_time_series_fname,
        west, south, east, north,
    )
    print(f"SIF: Computed time series values for {n_dates} granules and saved to {sif_csv_path}")
else:
    sif_csv_path = f"data/{sif_time_series_fname}"

if not os.path.exists(f"data/{swdi_time_series_fname}"):
    swdi_csv_path = compute_swdi_timeseries(
        start.strftime("%Y-%m-%d"),
        end.strftime("%Y-%m-%d"),
        swdi_time_series_fname,
        (west, south, east, north),
    )
    print(f"SWDI: Saved time series values to {swdi_csv_path}")
else:
    swdi_csv_path = f"data/{swdi_time_series_fname}"

In [ ]:
import csv
from datetime import datetime

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms

# Read the saved CSV
plot_dates = []
plot_zscore: list[float] = []
plot_sifrci: list[float] = []
with open(sif_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        pt_date = datetime.strptime(row["date"], "%Y-%m-%d") # noqa: DTZ007
        plot_dates.append(pt_date)
        plot_zscore.append(float(row["zscore"]))
        plot_sifrci.append(float(row["sif_rci"]))

plot2_dates = []
plot2_swdi: list[float] = []
with open(swdi_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        pt_date = datetime.strptime(row["time"], "%Y-%m-%d") # noqa: DTZ007
        # Our time series of SWDI is slightly longer, so skip dates that aren't
        # in the SIF time series
        if pt_date < plot_dates[0] or pt_date > plot_dates[-1]:
            continue
        plot2_dates.append(pt_date)
        plot2_swdi.append(float(row["swdi"]))


# As before, we will highlight the period where the SIF-RCI signal
# was persistently below 0
flash_drought = [r < -0.05 for r in plot_sifrci]

# Like in the SWDI plot from the first notebook, we will highlight the period
# where the SWDI signal was persistently below -5
moderate_fd_thresh = -3
fd_thresh = -5
flash_drought_swdi = [r < fd_thresh for r in plot2_swdi]

def add_crop_calendar(ax: plt.Axes, label: bool = True) -> None:
    """Cross-hatch the usual planting and harvest windows and label them on the plot."""
    ax.axvspan(planting_start, planting_end, facecolor="none", edgecolor="grey", # type: ignore
               hatch="///", linewidth=0.0, alpha=0.2, zorder=0)
    ax.axvspan(harvest_start, harvest_end, facecolor="none", edgecolor="grey", # type: ignore
               hatch="///", linewidth=0.0, alpha=0.2, zorder=0)
    if label:
        # x in data coordinates, y in axes-fraction coordinates
        trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
        ax.text(planting_start + (planting_end - planting_start) / 2, 0.98, # type: ignore
                "Planting /\nLeaf Development", transform=trans, ha="center",
                va="top", fontsize=10, color="black")
        ax.text(harvest_start + (harvest_end - harvest_start) / 2, 0.98, # type: ignore
                "Harvest", transform=trans, ha="center", va="top",
                fontsize=10, color="black")


# Plot the time series as two stacked subplots sharing the same x-axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Shade the crop calendar behind both subplots (labelled on the top subplot only)
add_crop_calendar(ax1)
add_crop_calendar(ax2, label=False)

# First subplot: SIF Standardized Anomaly (z-score) and SIF-RCI
ax1.axhline(0, color="black", linestyle=":", linewidth=1)
ax1.plot(plot_dates, plot_zscore, color="darkgreen", linewidth=0.5, label="SIF Standardized Anomaly")
ax1.plot(plot_dates, plot_sifrci, marker="o", color=(0.77, 0.64, 0.52), linewidth=2, markersize=4, label="SIF-RCI")
ax1.fill_between(plot_dates, plot_sifrci, 0, where=flash_drought, color="brown", alpha=0.1)
ax1.set_title(plot_title)
ax1.legend()

# Second subplot: SWDI with threshold levels
ax2.set_ylim(-10, 0)
ax2.axhline(moderate_fd_thresh, color="goldenrod", linestyle=":", linewidth=1, label="Moderate deficit")
ax2.axhline(fd_thresh, color="red", linestyle=":", linewidth=1, label="Severe deficit")
ax2.plot(plot2_dates, plot2_swdi, color="darkblue", linewidth=1.5, label=f"{plot_year_str} SWDI")
ax2.fill_between(plot2_dates, plot2_swdi, fd_thresh, where=flash_drought_swdi, color="brown", alpha=0.1)
ax2.set_ylabel("SWDI")
ax2.set_title(f"SWDI ({plot_year_str})")
ax2.legend()

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()